# 04 · Reevaluación del modelo de producción actual (sin reentrenar)
Modelo desplegado en la app (`modelo_S4_lstm.keras`: apilado 64/64, WFL α=1.5 β=100, LOOKBACK 360, HORIZON 20, solo S4 + cíclicas) sobre su **propio split** (2022–2026). Responde con evidencia a dos preguntas del especialista: **¿cuál es la precisión/tasa de falsas alarmas?** y **¿supera significativamente a la persistencia?** (bootstrap por días + Diebold–Mariano).

In [4]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from s4.config import cargar_config, variante, ruta
import json, joblib, tensorflow as tf
from pathlib import Path
from s4.pipeline import preparar_datos, baselines, calibrar_umbral, evaluar_y_registrar
from s4.windows import preparar_particion
from s4 import evaluate as ev, stats
cfg = cargar_config('../config/s4_completo_2022_2026.yaml')
DEPLOY = Path(cfg["rutas"]["base"]) / "app/produccion_actual"      # <-- carpeta de la app
conf = json.load(open(DEPLOY / "config.json"))
esc = joblib.load(DEPLOY / "SCALER_S4_app.pkl"); esc.columnas_ = conf["FEATURES_COLS"]
modelo = tf.keras.models.load_model(DEPLOY / "modelo_S4_lstm.keras", compile=False)
datos = preparar_datos(cfg, usar_precursores=False)

S4 reconstruido: 1,891,672 obs | 2022-01-01 00:00:00 -> 2026-09-13 14:15:00 | minutos reconstruidos: 15,457 (0.82 %) | flags: ['Fix0', 'Fix1', 'Fix3', 'Fix4', 'Fix_Pico']
✓ Auditoría de fuga: orden estricto TRAIN < VAL < TEST, sin solapamientos y ningún evento partido entre particiones.
 particion     inicio                 fin  observaciones  dias  dias_activos  pct_dias_activos  eventos  minutos_evento
     TRAIN 2022-01-01 2025-01-30 23:59:00        1135465   795            83             10.44      477             826
VALIDACIÓN 2025-01-31 2025-09-14 23:59:00         301928   211            28             13.27      197             402
      TEST 2025-09-15 2026-09-13 14:15:00         454279   319            64             20.06      523             980


In [7]:
L, H, F = conf["LOOKBACK"], conf["HORIZON"], conf["FEATURES_COLS"]
parts = {k: preparar_particion(k, datos[k], esc, F, L, H, hora_corte_ut=0) for k in ("val", "test")}
from s4.train import predecir
pred = {k: predecir(modelo, parts[k], esc) for k in parts}
base = {k: baselines(cfg, datos, {**parts, "train": None}, k) for k in parts}

val       : 298,517 ventanas | X=(360,5) | Y=(20) | objetivos reconstruidos: 2.67 %
test      : 446,699 ventanas | X=(360,5) | Y=(20) | objetivos reconstruidos: 2.35 %


2026-10-06 16:59:41.919437: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 450.00MiB (rounded to 471859456)requested by op LSTM_apilado_LB360_H20_1/lstm_1/CudnnRNNV3
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2026-10-06 16:59:41.919457: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1058] BFCAllocator dump for GPU_0_bfc
2026-10-06 16:59:41.919461: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1065] Bin (256): 	Total Chunks: 34, Chunks in use: 34. 8.5KiB allocated for chunks. 8.5KiB in use in bin. 1.2KiB client-requested in use in bin.
2026-10-06 16:59:41.919464: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1065] Bin (512): 	Total Chunks: 0, Chunks in use: 0. 0B allocated for chunks. 0B in use in bin. 0B client-requested in u

InternalError: Graph execution error:

Detected at node LSTM_apilado_LB360_H20_1/lstm_1/CudnnRNNV3 defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/home/soporte/anaconda3/envs/s4_env/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/soporte/anaconda3/envs/s4_env/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/soporte/anaconda3/envs/s4_env/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 739, in start

  File "/home/soporte/anaconda3/envs/s4_env/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 205, in start

  File "/home/soporte/anaconda3/envs/s4_env/lib/python3.12/asyncio/base_events.py", line 645, in run_forever

  File "/home/soporte/anaconda3/envs/s4_env/lib/python3.12/asyncio/base_events.py", line 1999, in _run_once

  File "/home/soporte/anaconda3/envs/s4_e

_allocator.cc:1114] InUse at 73a95e03e000 of size 1792 next 59
2026-10-06 16:59:41.919549: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1114] InUse at 73a95e03e700 of size 1024 next 45
2026-10-06 16:59:41.919550: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1114] InUse at 73a95e03eb00 of size 1024 next 50
2026-10-06 16:59:41.919550: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1114] InUse at 73a95e03ef00 of size 72192 next 30
2026-10-06 16:59:41.919551: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1114] InUse at 73a95e050900 of size 1024 next 17
2026-10-06 16:59:41.919552: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1114] InUse at 73a95e050d00 of size 9216 next 36
2026-10-06 16:59:41.919553: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1114] InUse at 73a95e053100 of size 16384 next 46
2026-10-06 16:59:41.919554: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1114] InUse at 73a95e057100 of size 5120 next 51
2026

In [ ]:
# 1) Umbral de alerta del modelo, calibrado en VALIDACIÓN (máximo HSS)
u_modelo, tabla_u = calibrar_umbral(cfg, parts["val"], pred["val"])
print("Umbral de alerta elegido en validación:", u_modelo); tabla_u

In [ ]:
# 2) Evaluación en TEST (una sola vez)
filas = [evaluar_y_registrar(cfg, parts["test"], base["test"]["Persistencia"], "Persistencia", "reevaluacion"),
         evaluar_y_registrar(cfg, parts["test"], base["test"]["Climatología diaria"], "Climatología", "reevaluacion"),
         evaluar_y_registrar(cfg, parts["test"], pred["test"], "LSTM apilado + WFL (u=0.6)", "reevaluacion"),
         evaluar_y_registrar(cfg, parts["test"], pred["test"], f"LSTM apilado + WFL (u={u_modelo})", "reevaluacion", umbral_alerta=u_modelo)]
cols = ["modelo","RMSE_global","RMSE_evento","RMSE_calma","sesgo_evento","POD","Precision","FAR","CSI","HSS","POD_inicio","antelacion_media_min"]
tabla = pd.DataFrame(filas)[cols]; tabla.to_csv(ruta(cfg,"reportes")/"tables/tabla_reevaluacion_produccion.csv", index=False); tabla.round(4)

In [ ]:
# 3) ¿En qué fase gana el modelo? (inicio vs sostenido)
pd.concat({"Persistencia": ev.metricas_por_fase(parts["test"], base["test"]["Persistencia"]),
           "LSTM": ev.metricas_por_fase(parts["test"], pred["test"], umbral_alerta=u_modelo)})

In [ ]:
# 4) Falsas alarmas: ¿son "casi eventos"? (pregunta del especialista)
ev.analisis_falsas_alarmas(parts["test"], pred["test"], umbral_alerta=u_modelo)

In [ ]:
# 5) Significancia frente a persistencia: bootstrap por DÍAS + Diebold–Mariano
p = parts["test"]; m = ~p.objetivo_reconstruido
args = (p.y_real, base["test"]["Persistencia"], pred["test"], p.dia)
print("RMSE evento :", stats.bootstrap_bloques(*args, stats.m_rmse_evento(), mascara=m))
print("RMSE global :", stats.bootstrap_bloques(*args, stats.m_rmse_global, mascara=m))
print("Diebold–Mariano (MSE):", stats.diebold_mariano(*args, mascara=m))

**Lectura:** diferencia > 0 con IC que no cruza 0 ⇒ la LSTM es significativamente mejor que la persistencia en esa métrica. Reportar ambas (evento y global): el trade-off es parte del resultado.